In [2]:
# Cell 1▶ Imports, data load, counters
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import accumulate
from collections import Counter

# load your processed CSV (notebooks/ is sibling to data/)
DF = pd.read_csv(
    "../data/processed/molecules_processed.csv",
    converters={
        "tokens_char":    lambda s: json.loads(s.replace("'", '"')),
        "tokens_regex":   lambda s: json.loads(s.replace("'", '"')),
        "tokens_selfies": lambda s: json.loads(s.replace("'", '"')),
    }
)

def make_counter(col):
    return Counter(tok for row in DF[col] for tok in row)

CNT = {
    "char":    make_counter("tokens_char"),
    "regex":   make_counter("tokens_regex"),
    "selfies": make_counter("tokens_selfies"),
}
CNT["union"] = CNT["char"] + CNT["regex"] + CNT["selfies"]

# Cell 2▶ Interactive widget & plotting function
from ipywidgets import interact, Dropdown, IntSlider

def viz_tokens(token_type: str, topn: int, mode: str):
    """
    token_type: one of 'char', 'regex', 'selfies', 'union'
    topn: how many top tokens for bar / coverage cutoff
    mode: 'bar' | 'zipf' | 'coverage'
    """
    counter = CNT[token_type]
    # prep data
    ranks, freqs = zip(*counter.most_common())
    # bar chart
    if mode == "bar":
        toks, cnts = zip(*counter.most_common(topn))
        colors = plt.get_cmap("viridis")(np.linspace(0,1,topn))
        plt.figure(figsize=(8,4))
        plt.bar(toks, cnts, color=colors)
        plt.title(f"Top {topn} {token_type} tokens")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

    # Zipf plot
    elif mode == "zipf":
        plt.figure(figsize=(6,4))
        plt.loglog(range(1,len(freqs)+1), freqs, marker=".")
        plt.title(f"{token_type} tokens (Zipf)")
        plt.xlabel("Rank")
        plt.ylabel("Frequency")
        plt.grid(True, which="both", ls=":")
        plt.tight_layout()
        plt.show()

    # Cumulative coverage
    elif mode == "coverage":
        total = sum(freqs)
        cum = [s/total for s in accumulate(freqs)]
        plt.figure(figsize=(6,4))
        plt.plot(range(1,len(cum)+1), cum, marker=".")
        plt.axvline(topn, color="red", linestyle="--")
        cov = cum[topn-1]*100
        plt.text(topn, cum[topn-1], f"{cov:.1f}%", va="bottom", ha="right")
        plt.xscale("log")
        plt.title(f"{token_type} tokens coverage")
        plt.xlabel("Top-K tokens")
        plt.ylabel("Cumulative fraction")
        plt.grid(True, which="both", ls=":")
        plt.tight_layout()
        plt.show()

# launch the interactive UI
interact(
    viz_tokens,
    token_type=Dropdown(options=["char","regex","selfies","union"], value="char", description="Type"),
    topn=IntSlider(min=5, max=200, step=5, value=20, description="Top-N"),
    mode=Dropdown(options=["bar","zipf","coverage"], value="bar", description="Mode")
)

interactive(children=(Dropdown(description='Type', options=('char', 'regex', 'selfies', 'union'), value='char'…

<function __main__.viz_tokens(token_type: str, topn: int, mode: str)>